# Selective interference simulation

Simulate the trauma-film + reminder + interference design and extract intrusion and recognition summaries for a single parameter set.

## What this notebook does

- Encodes a toy trauma-film sequence, applies delay + reminder, then branches into control vs interference.
- Runs vigilance-intrusion (VIT) cues and recognition cues/targets from both branches.
- Summarizes intrusions (mean per cue) and recognition (hit/FA/accuracy) to inspect the selective-interference pattern.
- Parameters are all in the tagged cell below so you can override with papermill.


## How to use

1. Adjust the parameters cell (tagged `parameters`) or override via papermill.
2. Run the notebook; it builds the encoded model, applies reminder/interference, and simulates VIT and recognition blocks.
3. Read the printed summaries to see control vs interference differences for intrusions and recognition.


In [1]:
import matplotlib.pyplot as plt
import os

import jax.numpy as jnp
from jax import lax, random

from jaxcmr.models.cmr import CMR
from jaxcmr.experimental.selective_interference import (
    apply_reminder_and_interference,
    cue_context,
    drift_context,
    simulate_recognition_block,
    simulate_vigilance_block,
)

def save_figure(figure_dir, figure_str, suffix=None):
    plt.tight_layout()
    if not figure_str:
        plt.show()
        return
    os.makedirs(figure_dir, exist_ok=True)
    suffix_str = f"_{suffix}" if suffix else ""
    figure_path = os.path.join(figure_dir, f"{figure_str}{suffix_str}.png")
    plt.savefig(figure_path, bbox_inches="tight", dpi=600)
    plt.show()


Parameters

In [2]:

# Run configuration
seed = 0

# Item pool
list_length = 14
film_ids = jnp.arange(1, 12)
foil_ids = jnp.array([12, 13, 14])
reminder_id = 1

# Delay and reminder/interference
delay_steps = 5
delay_drift_rate = 0.15
reminder_drift_rate = 0.3
interference_steps = 10
interference_drift_rate = 0.4

# Vigilance-intrusion task
vit_cues = jnp.array([1, 2, 5, 12, 13], dtype=int)
vit_cue_types = jnp.array([0, 0, 0, 1, 1], dtype=int)  # 0 film, 1 foil
vit_cue_drift_rate = 0.25
max_intrusions = 3

# Recognition task
recog_cues = jnp.array([1, 4, 12, 13], dtype=int)
recog_targets = jnp.array([2, 7, 12, 3], dtype=int)
recog_target_types = jnp.array([0, 0, 1, 0], dtype=int)  # 0 film, 1 foil
recog_cue_drift_rate = 0.25
decision_threshold = 0.15

# CMR parameters
params = {
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "shared_support": 0.05,
    "item_support": 0.25,
    "primacy_scale": 1.0,
    "primacy_decay": 0.6,
    "learning_rate": 0.5,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "choice_sensitivity": 0.6,
    "learn_after_context_update": True,
    "allow_repeated_recalls": False,
}
figure_dir = "results/figures"
figure_str = ""


In [3]:

# Build film sequence and encode
film_sequence = jnp.array(film_ids, dtype=int)
base_model = CMR(list_length, params)
encoded = lax.fori_loop(0, film_sequence.size, lambda i, m: m.experience(film_sequence[i]), base_model)

# Delay
post_delay = drift_context(encoded, delay_drift_rate, delay_steps)

# Cue feature lookup
def probe_fn(idx):
    return post_delay.items[idx - 1]

# Reminder and interference branches
control_model, interference_model = apply_reminder_and_interference(
    post_delay,
    reminder_id,
    reminder_drift_rate,
    interference_steps,
    interference_drift_rate,
    probe_fn,
)


In [4]:

# Simulate tasks
rng = random.PRNGKey(seed)
rng, vit_control_key, vit_interference_key, rec_control_key, rec_interference_key = random.split(rng, 5)

control_vit = simulate_vigilance_block(
    control_model,
    vit_cues,
    vit_cue_drift_rate,
    max_intrusions,
    vit_control_key,
    probe_fn,
)
interference_vit = simulate_vigilance_block(
    interference_model,
    vit_cues,
    vit_cue_drift_rate,
    max_intrusions,
    vit_interference_key,
    probe_fn,
)

control_recognition = simulate_recognition_block(
    control_model,
    recog_cues,
    recog_targets,
    recog_cue_drift_rate,
    decision_threshold,
    rec_control_key,
    probe_fn,
)
interference_recognition = simulate_recognition_block(
    interference_model,
    recog_cues,
    recog_targets,
    recog_cue_drift_rate,
    decision_threshold,
    rec_interference_key,
    probe_fn,
)


## Reading the outputs

- Intrusions: mean count per cue, split by film/foil cues. A selective effect shows a large drop for film cues in the interference branch.
- Recognition: hit rate, false-alarm rate, and accuracy. A selective effect shows little/no change here while intrusions drop.
- Tweak drift rates, thresholds, or cue/target lists to explore robustness.


In [5]:

# Summaries

def summarize_intrusions(recalls, cue_types):
    counts = (recalls != 0).sum(axis=1)
    film_mask = cue_types == 0
    foil_mask = cue_types == 1
    film_total = jnp.sum(film_mask)
    foil_total = jnp.sum(foil_mask)
    film_mean = jnp.where(film_total > 0, jnp.sum(counts * film_mask) / film_total, 0.0)
    foil_mean = jnp.where(foil_total > 0, jnp.sum(counts * foil_mask) / foil_total, 0.0)
    return {
        "film_mean": float(film_mean),
        "foil_mean": float(foil_mean),
        "overall_mean": float(jnp.mean(counts)),
    }


def summarize_recognition(decisions, target_types):
    film_mask = target_types == 0
    foil_mask = target_types == 1
    film_total = jnp.sum(film_mask)
    foil_total = jnp.sum(foil_mask)
    film_hits = jnp.where(film_total > 0, jnp.sum(decisions * film_mask) / film_total, 0.0)
    foil_fas = jnp.where(foil_total > 0, jnp.sum(decisions * foil_mask) / foil_total, 0.0)
    return {
        "hit_rate": float(film_hits),
        "fa_rate": float(foil_fas),
        "accuracy": float(jnp.mean(decisions)),
    }

control_intrusions = summarize_intrusions(control_vit, vit_cue_types)
interference_intrusions = summarize_intrusions(interference_vit, vit_cue_types)

control_recog_summary = summarize_recognition(control_recognition, recog_target_types)
interference_recog_summary = summarize_recognition(interference_recognition, recog_target_types)

print("Intrusions (mean counts per cue)")
print({"control": control_intrusions, "interference": interference_intrusions})
print()
print("Recognition")
print({"control": control_recog_summary, "interference": interference_recog_summary})


Intrusions (mean counts per cue)
{'control': {'film_mean': 1.6666666269302368, 'foil_mean': 3.0, 'overall_mean': 2.200000047683716}, 'interference': {'film_mean': 2.0, 'foil_mean': 3.0, 'overall_mean': 2.4000000953674316}}

Recognition
{'control': {'hit_rate': 0.0, 'fa_rate': 0.0, 'accuracy': 0.0}, 'interference': {'hit_rate': 0.0, 'fa_rate': 0.0, 'accuracy': 0.0}}
